In [ ]:
# Cell 1: Core imports for knowledge graph creation
import os
import sys
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path to access scripts
sys.path.append('..')

# Knowledge graph specific imports
from knowledge_graph_maker import GraphMaker, Ontology
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# LLM and NLP imports
import openai
from openai import OpenAI

# Text processing
import re
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Set
import json
from tqdm import tqdm
import time
from datetime import datetime
import pickle

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ All packages imported successfully!")

In [ ]:
# Cell 2: Load environment variables and configure
load_dotenv(override=True)

# Configure matplotlib for better fonts
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print("🔧 Setting up configuration...")

# Basic configuration
class Config:
    def __init__(self):
        # API Keys
        self.openai_api_key = os.getenv('OPENAI_API_KEY')
        
        # Model settings
        self.llm_model = os.getenv('DEFAULT_LLM_MODEL', 'gpt-4o-mini')
        self.embedding_model = os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small')
        
        # Graph settings
        self.max_entities = int(os.getenv('MAX_ENTITIES_PER_CHUNK', 20))
        self.max_relationships = int(os.getenv('MAX_RELATIONSHIPS_PER_CHUNK', 30))
        self.chunk_size = int(os.getenv('CHUNK_SIZE', 1000))
        self.chunk_overlap = int(os.getenv('CHUNK_OVERLAP', 200))
        
        # Create directories
        self.ensure_directories()
    
    def ensure_directories(self):
        """Create necessary directories"""
        directories = ['../knowledge_graphs', '../graph_visualizations', '../graph_data']
        for directory in directories:
            if not os.path.exists(directory):
                os.makedirs(directory)
                print(f"📁 Created directory: {directory}")

# Initialize configuration
config = Config()

# Initialize LLM clients
def initialize_llm_clients():
    """Initialize LLM clients based on available API keys"""
    clients = {}
    
    if config.openai_api_key:
        clients['openai'] = OpenAI(api_key=config.openai_api_key)
        print("✅ OpenAI client initialized")
        print(f"🎯 Using model: {config.llm_model}")
    else:
        print("⚠️  No OpenAI API key found. Please set OPENAI_API_KEY in .env file")
        print("💡 Get your API key from: https://platform.openai.com/api-keys")
    
    return clients

llm_clients = initialize_llm_clients()

print("\n🎯 Setup complete! Ready for knowledge graph creation.")

In [ ]:
# Cell 3: Create ontology for Northern Ireland education discourse
from knowledge_graph_maker import Ontology

def create_ni_education_ontology():
    """
    Create ontology for Northern Ireland education and historical discourse analysis
    """
    
    ontology = Ontology(
        # Labels of the entities to be extracted
        labels=[
            {"Person": "Person name without any adjectives. A person may be referenced by their name or using a pronoun"},
            {"Object": "Do not add the definite article 'the' in the object name"},
            {"Event": "Historical or educational event involving multiple people. Do not include qualifiers or verbs"},
            "Place",
            "Document",
            "Organisation",
            "Action",
            "Emotions",
            {"Miscellaneous": "Any important concept that cannot be categorised with any other given label"},
        ],
        
        # Relationships between entities
        relationships=[
            "Relation between any pair of Entities",
        ]
    )
    
    return ontology

# Create the ontology
ni_ontology = create_ni_education_ontology()
print("🎯 Northern Ireland Education ontology created successfully!")
print(f"📊 Entity types: {len(ni_ontology.labels)}")
print(f"🔗 Relationship types: {len(ni_ontology.relationships)}")

print("\n🏷️ Entity types:")
for entity in ni_ontology.labels:
    print(f"  • {entity}")

In [ ]:
# Cell 4: Load and inspect the cleaned text data
def load_ni_education_data(filepath):
    """Load and inspect the Northern Ireland education data"""
    print(f"📂 Loading data: {filepath}")
    df = pd.read_csv(filepath)
    
    # Display basic information
    print(f"\n📊 Dataset size: {df.shape}")
    print(f"\n📋 Columns: {df.columns.tolist()}")
    print(f"\n🔍 Missing values:\n{df.isnull().sum()}")
    
    # Show religious group distribution
    print(f"\n🏛️ Religious group distribution:")
    print(df['religious_group'].value_counts())
    
    # Show file type distribution
    print(f"\n📚 File type distribution:")
    print(df['file_type'].value_counts())
    
    # Sample the data
    print(f"\n📝 Sample records:")
    print(df[['filename', 'religious_group', 'file_type', 'content_length']].head())
    
    return df

# Load the data
df_ni = load_ni_education_data('../outputs/cleaned_text_data.csv')

# Quick statistics
print(f"\n📈 Content statistics:")
print(f"  • Average content length: {df_ni['content_length'].mean():.0f} characters")
print(f"  • Average word count: {df_ni['word_count'].mean():.0f} words")
print(f"  • Total documents: {len(df_ni)}")
print(f"  • Option1 documents: {len(df_ni[df_ni['religious_group'] == 'option1'])}")
print(f"  • Option2 documents: {len(df_ni[df_ni['religious_group'] == 'option2'])}")

In [ ]:
# Cell 5: Text preprocessing for Northern Ireland education texts
class NIEducationPreprocessor:
    """
    Preprocessor for Northern Ireland education textbook analysis
    """
    
    def __init__(self, chunk_size=1000, chunk_overlap=200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        
        print(f"Text chunking configured: size={chunk_size}, overlap={chunk_overlap}")
    
    def clean_text_for_kg(self, text: str) -> str:
        """Clean text for knowledge graph extraction"""
        if pd.isna(text) or text.strip() == '':
            return ""
            
        text = str(text).strip()
        
        # Remove excessive whitespace
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\n+', '\n', text)
        
        # Keep text that's meaningful for relationship extraction
        if len(text.strip()) < 50:
            return ""
            
        return text.strip()
    
    def chunk_text_semantic(self, text: str) -> List[Dict]:
        """
        Create semantic chunks with metadata
        """
        if not text or len(text) < 100:
            return [{'raw_text': text}] if text else []
        
        # Split by sentences
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() + '.' for s in sentences if s.strip()]
        
        chunks = []
        current_chunk = ""
        
        for sentence in sentences:
            if len(current_chunk + sentence) > self.chunk_size:
                if current_chunk:
                    chunks.append({
                        'raw_text': current_chunk.strip(),
                        'char_count': len(current_chunk)
                    })
                    
                    # Start new chunk with overlap
                    overlap_sentences = current_chunk.split('.')[-2:]
                    current_chunk = '.'.join(overlap_sentences) + '.' + sentence
                else:
                    current_chunk = sentence
            else:
                current_chunk += sentence
        
        # Add the last chunk
        if current_chunk.strip():
            chunks.append({
                'raw_text': current_chunk.strip(),
                'char_count': len(current_chunk)
            })
        
        return chunks

# Initialize preprocessor
preprocessor = NIEducationPreprocessor(chunk_size=1000, chunk_overlap=200)
print("Text preprocessor ready")

In [ ]:
# Cell 6: Rate limiting manager
class SimpleRateLimitManager:
    """Simplified rate limiting with cost tracking"""
    
    def __init__(self):
        self.request_count = 0
        self.start_time = datetime.now()
        self.total_cost = 0.0
    
    def safe_wait(self):
        """1-second delay between requests"""
        time.sleep(1.0)
        self.request_count += 1
        
        if self.request_count % 10 == 0:
            elapsed = (datetime.now() - self.start_time).seconds
            print(f"  Processed {self.request_count} requests in {elapsed}s")
    
    def estimate_cost(self, input_tokens, output_tokens):
        """Estimate cost for GPT-4o-mini"""
        cost = (input_tokens * 0.15 + output_tokens * 0.60) / 1000000
        self.total_cost += cost
        return cost

rate_limiter = SimpleRateLimitManager()
print("Rate limiter initialized")

In [ ]:
# Cell 7: Knowledge extraction with JSON parsing
from dataclasses import dataclass

@dataclass
class KnowledgeTriple:
    """Structure for storing extracted knowledge triples"""
    subject: str
    predicate: str
    object: str
    subject_type: str
    object_type: str
    confidence: float
    source_corpus: str
    chunk_id: str

def extract_json_from_response(content: str) -> str:
    """Extract JSON content from markdown-wrapped responses"""
    json_pattern = r'```json\s*(.*?)\s*```'
    match = re.search(json_pattern, content, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    json_array_pattern = r'(\[.*\])'
    match = re.search(json_array_pattern, content, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    return content.strip()

class NIKnowledgeExtractor:
    """Knowledge extractor for Northern Ireland education texts"""
    
    def __init__(self, ontology, client: OpenAI, model: str = "gpt-4o-mini"):
        self.ontology = ontology
        self.client = client
        self.model = model
        self.entity_types = ontology.labels
        self.relationship_types = ontology.relationships
        
        self.extraction_stats = {
            'total_chunks': 0,
            'successful_extractions': 0,
            'failed_extractions': 0,
            'total_triples': 0,
            'api_calls': 0,
            'total_cost': 0.0
        }
        
        print(f"Knowledge extractor initialized with model: {model}")
        print(f"Entity types: {len(self.entity_types)}")
        print(f"Relationship types: {len(self.relationship_types)}")
    
    def create_extraction_prompt(self, text_chunk: str, corpus_name: str) -> str:
        """Create prompt for NI education discourse analysis"""
        
        entity_types_str = ", ".join([str(e) for e in self.entity_types])
        
        prompt = f"""You are an expert in analyzing Northern Ireland educational texts and historical discourse.
Extract structured knowledge relationships from the following text.

**Text Source**: {corpus_name}
**Analysis Focus**: Educational perspectives on Northern Ireland history, focusing on how different communities present historical events, people, and concepts.

**Text Content**:
{text_chunk}

**Extraction Rules**:
1. **Entity Types**: Only extract entities of these types:
   {entity_types_str}

2. **Focus Areas**:
   - Historical events and their interpretations
   - Key figures and their roles
   - Places and their significance
   - Educational narratives and perspectives
   - Emotional and evaluative language
   - Organizational relationships

3. **Output Format**: JSON array where each relationship contains:
   - "subject": subject entity
   - "subject_type": subject entity type
   - "predicate": relationship type
   - "object": object entity
   - "object_type": object entity type
   - "confidence": confidence score (0.0-1.0)

Output only the JSON array, ensuring all types are from the specified list."""

        return prompt
    
    def extract_knowledge_from_chunk(self, text_chunk: str, corpus_name: str, 
                                   chunk_id: str) -> List[KnowledgeTriple]:
        """Extract knowledge triples from a single text chunk"""
        
        prompt = self.create_extraction_prompt(text_chunk, corpus_name)
        
        try:
            rate_limiter.safe_wait()
            
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are an expert in extracting structured knowledge from educational texts."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=1500,
                temperature=0.1
            )
            
            self.extraction_stats['api_calls'] += 1
            
            content = response.choices[0].message.content
            json_content = extract_json_from_response(content)
            
            if hasattr(response, 'usage'):
                cost = rate_limiter.estimate_cost(
                    response.usage.prompt_tokens,
                    response.usage.completion_tokens
                )
                self.extraction_stats['total_cost'] += cost
            
            try:
                relations_data = json.loads(json_content)
                if isinstance(relations_data, dict) and 'relations' in relations_data:
                    relations_data = relations_data['relations']
                elif not isinstance(relations_data, list):
                    relations_data = []
            except json.JSONDecodeError:
                self.extraction_stats['failed_extractions'] += 1
                return []
            
            triples = []
            for relation in relations_data:
                try:
                    triple = KnowledgeTriple(
                        subject=relation['subject'],
                        predicate=relation['predicate'],
                        object=relation['object'],
                        subject_type=relation['subject_type'],
                        object_type=relation['object_type'],
                        confidence=float(relation.get('confidence', 0.8)),
                        source_corpus=corpus_name,
                        chunk_id=chunk_id
                    )
                    triples.append(triple)
                except (KeyError, ValueError):
                    continue
            
            self.extraction_stats['successful_extractions'] += 1
            self.extraction_stats['total_triples'] += len(triples)
            
            return triples
            
        except Exception as e:
            print(f"Extraction failed for {chunk_id}: {e}")
            self.extraction_stats['failed_extractions'] += 1
            return []
    
    def extract_from_corpus(self, chunks: List[str], metadata: List[Dict], 
                          corpus_name: str, max_chunks: int = None,
                          save_progress: bool = True) -> List[KnowledgeTriple]:
        """Extract knowledge from an entire corpus"""
        print(f"\nStarting extraction from {corpus_name}...")
        print(f"Total chunks: {len(chunks)}")
        
        if max_chunks:
            chunks = chunks[:max_chunks]
            metadata = metadata[:max_chunks]
            print(f"Limited to: {max_chunks} chunks")
        
        all_triples = []
        self.extraction_stats['total_chunks'] += len(chunks)
        
        for i, (chunk, meta) in enumerate(tqdm(zip(chunks, metadata), 
                                              total=len(chunks), 
                                              desc=f"Extracting {corpus_name}")):
            
            chunk_id = f"{corpus_name}_{i}_{meta.get('original_index', i)}"
            triples = self.extract_knowledge_from_chunk(chunk, corpus_name, chunk_id)
            all_triples.extend(triples)
            
            if (i + 1) % 25 == 0:
                print(f"  Processed {i+1}/{len(chunks)}, extracted {len(all_triples)} relations")
                if save_progress:
                    self.save_progress(all_triples, f"{corpus_name}_progress_{i+1}")
        
        print(f"Extraction complete for {corpus_name}!")
        print(f"  Extracted: {len(all_triples)} knowledge relations")
        print(f"  Cost: ${self.extraction_stats['total_cost']:.3f}")
        
        if save_progress:
            self.save_progress(all_triples, f"{corpus_name}_final")
        
        return all_triples
    
    def save_progress(self, triples: List[KnowledgeTriple], filename_suffix: str):
        """Save extraction progress"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        pickle_path = f"../graph_data/kg_triples_{filename_suffix}_{timestamp}.pkl"
        with open(pickle_path, 'wb') as f:
            pickle.dump(triples, f)
        print(f"Progress saved: {pickle_path}")
    
    def print_final_summary(self):
        """Print final extraction summary"""
        stats = self.extraction_stats
        print("\n" + "="*60)
        print("Final Knowledge Graph Extraction Summary")
        print("="*60)
        print(f"Total chunks: {stats['total_chunks']:,}")
        print(f"Successful: {stats['successful_extractions']:,}")
        print(f"Failed: {stats['failed_extractions']:,}")
        print(f"Success rate: {stats['successful_extractions']/(stats['total_chunks'])*100:.1f}%")
        print(f"Total relations: {stats['total_triples']:,}")
        print(f"API calls: {stats['api_calls']:,}")
        print(f"Total cost: ${stats['total_cost']:.3f}")
        print(f"Avg relations per chunk: {stats['total_triples']/max(stats['successful_extractions'], 1):.1f}")
        print("="*60)

# Initialize extractor
knowledge_extractor = NIKnowledgeExtractor(
    ontology=ni_ontology,
    client=llm_clients['openai'],
    model="gpt-4o-mini"
)

print("Knowledge extraction system ready")

In [ ]:
# Cell 8: Data preparation - Sample and chunk texts by religious group
def prepare_ni_corpus_data(df, sample_sizes={'option1': 100, 'option2': 100, 'both': 30}):
    """
    Prepare Northern Ireland education corpus data for knowledge extraction
    Samples from each religious group and creates chunks
    """
    print("📄 Preparing corpus data for knowledge graph extraction...")
    
    corpora_data = {}
    
    # Process each religious group
    for group, sample_size in sample_sizes.items():
        print(f"\n{'='*60}")
        print(f"Processing {group.upper()} corpus")
        print(f"{'='*60}")
        
        # Filter data for this group
        group_df = df[df['religious_group'] == group].copy()
        
        # Remove rows with missing content
        group_df = group_df.dropna(subset=['cleaned_content'])
        group_df = group_df[group_df['cleaned_content'].str.len() > 100]
        
        print(f"  Available documents: {len(group_df)}")
        
        # Sample documents
        if len(group_df) > sample_size:
            group_sample = group_df.sample(n=sample_size, random_state=42)
            print(f"  Sampled: {sample_size} documents")
        else:
            group_sample = group_df
            print(f"  Using all {len(group_sample)} documents")
        
        # Create chunks from sampled documents
        chunks = []
        metadata = []
        
        for idx, row in group_sample.iterrows():
            # Clean text
            cleaned_text = preprocessor.clean_text_for_kg(row['cleaned_content'])
            
            if cleaned_text:
                # Create semantic chunks
                text_chunks = preprocessor.chunk_text_semantic(cleaned_text)
                
                for chunk_idx, chunk_data in enumerate(text_chunks):
                    if len(chunk_data['raw_text']) > 100:
                        chunks.append(chunk_data['raw_text'])
                        metadata.append({
                            'corpus': group,
                            'original_index': idx,
                            'chunk_index': chunk_idx,
                            'filename': row['filename'],
                            'file_type': row['file_type'],
                            'char_count': chunk_data['char_count']
                        })
        
        corpora_data[group] = {
            'chunks': chunks,
            'metadata': metadata,
            'name': f"{group.upper()} Perspective",
            'sample_size': len(group_sample),
            'chunk_count': len(chunks)
        }
        
        print(f"  Generated {len(chunks)} text chunks")
        print(f"  Average chunk size: {np.mean([m['char_count'] for m in metadata]):.0f} chars")
    
    # Print summary
    print(f"\n{'='*60}")
    print("📊 Data Preparation Summary")
    print(f"{'='*60}")
    
    for name, data in corpora_data.items():
        print(f"{data['name']:20s}: {data['sample_size']:3d} docs → {data['chunk_count']:4d} chunks")
    
    total_chunks = sum(data['chunk_count'] for data in corpora_data.values())
    estimated_cost = total_chunks * 0.003
    estimated_time = total_chunks * 1.5 / 60
    
    print(f"\n💰 Estimated cost: ${estimated_cost:.2f}")
    print(f"⏱️  Estimated time: {estimated_time:.1f} minutes")
    
    return corpora_data

# Prepare the data
ni_corpora_data = prepare_ni_corpus_data(
    df_ni, 
    sample_sizes={
        'option1': 1000,  # Protestant/Unionist perspective
        'option2': 1000,  # Catholic/Nationalist perspective
        'both': 140       # Shared/neutral documents
    }
)

In [ ]:
# Cell 9: Cost estimation and execution planning
def estimate_extraction_costs(corpora_data):
    """
    Estimate costs and time for different extraction scales
    """
    print("💰 Cost and Time Estimation for Knowledge Graph Extraction")
    print("="*70)
    print("Based on GPT-4o-mini pricing ($0.15/1M input, $0.60/1M output)\n")
    
    total_chunks = sum(data['chunk_count'] for data in corpora_data.values())
    
    scenarios = [
        ("Small Test", 10, "10 chunks per corpus"),
        ("Medium Test", 30, "30 chunks per corpus"),
        ("Standard Run", 50, "50 chunks per corpus"),
        ("Full Extraction", None, "All chunks")
    ]
    
    print(f"{'Scenario':<20} {'Chunks':<10} {'Est. Cost':<12} {'Est. Time':<12} {'Description'}")
    print("-"*70)
    
    for name, chunks_per_corpus, desc in scenarios:
        if chunks_per_corpus:
            total = chunks_per_corpus * len(corpora_data)
        else:
            total = total_chunks
        
        cost = total * 0.003
        time_mins = total * 1.5 / 60
        
        print(f"{name:<20} {total:<10} ${cost:<11.2f} {time_mins:<11.1f}min {desc}")
    
    print("\n⚠️  Recommendations:")
    print("  • Start with Small Test (10 chunks) to verify extraction quality")
    print("  • Review sample results before scaling up")
    print("  • Use Standard Run (50 chunks) for balanced coverage")
    print("  • Full extraction may take 1+ hours and cost $10+")

estimate_extraction_costs(ni_corpora_data)

In [ ]:
# Cell 10: Small test extraction
def run_test_extraction(corpora_data, test_chunks=10):
    """
    Run a small test extraction to verify system is working
    """
    print(f"\n🧪 Running test extraction ({test_chunks} chunks per corpus)...")
    print("="*60)
    
    test_results = {}
    
    for corpus_name, data in corpora_data.items():
        print(f"\n📚 Testing {data['name']}...")
        
        # Extract from first few chunks
        test_triples = knowledge_extractor.extract_from_corpus(
            chunks=data['chunks'][:test_chunks],
            metadata=data['metadata'][:test_chunks],
            corpus_name=data['name'],
            max_chunks=test_chunks,
            save_progress=False
        )
        
        test_results[corpus_name] = test_triples
        
        print(f"  ✅ Test complete: {len(test_triples)} relations extracted")
        
        # Show sample results
        if test_triples:
            print(f"  📋 Sample relations:")
            for i, triple in enumerate(test_triples[:3]):
                print(f"    {i+1}. {triple.subject} ({triple.subject_type}) "
                      f"→ {triple.predicate} → "
                      f"{triple.object} ({triple.object_type}) "
                      f"[confidence: {triple.confidence:.2f}]")
    
    knowledge_extractor.print_final_summary()
    
    return test_results

# Run test extraction
print("🚀 Starting test extraction...")
test_results = run_test_extraction(ni_corpora_data, test_chunks=10)

In [ ]:
# Cell 17: Configure for full extraction
def prepare_full_extraction(df):
    """
    Prepare ALL data for full knowledge graph extraction
    """
    print("📄 Preparing FULL corpus data for extraction...")
    print("="*70)
    
    corpora_data = {}
    
    # Process each religious group - using ALL available data
    for group in ['option1', 'option2', 'both']:
        print(f"\n{'='*60}")
        print(f"Processing {group.upper()} corpus (FULL)")
        print(f"{'='*60}")
        
        # Filter data for this group
        group_df = df[df['religious_group'] == group].copy()
        
        # Remove rows with missing content
        group_df = group_df.dropna(subset=['cleaned_content'])
        group_df = group_df[group_df['cleaned_content'].str.len() > 100]
        
        print(f"  Total documents: {len(group_df)}")
        
        # Create chunks from ALL documents (no sampling)
        chunks = []
        metadata = []
        
        for idx, row in tqdm(group_df.iterrows(), total=len(group_df), desc=f"Processing {group}"):
            # Clean text
            cleaned_text = preprocessor.clean_text_for_kg(row['cleaned_content'])
            
            if cleaned_text:
                # Create semantic chunks
                text_chunks = preprocessor.chunk_text_semantic(cleaned_text)
                
                for chunk_idx, chunk_data in enumerate(text_chunks):
                    if len(chunk_data['raw_text']) > 100:
                        chunks.append(chunk_data['raw_text'])
                        metadata.append({
                            'corpus': group,
                            'original_index': idx,
                            'chunk_index': chunk_idx,
                            'filename': row['filename'],
                            'file_type': row['file_type'],
                            'char_count': chunk_data['char_count']
                        })
        
        corpora_data[group] = {
            'chunks': chunks,
            'metadata': metadata,
            'name': f"{group.upper()} Perspective",
            'doc_count': len(group_df),
            'chunk_count': len(chunks)
        }
        
        print(f"  ✅ Generated {len(chunks)} text chunks from {len(group_df)} documents")
        print(f"  📊 Average chunk size: {np.mean([m['char_count'] for m in metadata]):.0f} chars")
    
    # Print comprehensive summary
    print(f"\n{'='*70}")
    print("📊 FULL EXTRACTION DATA SUMMARY")
    print(f"{'='*70}")
    
    total_docs = 0
    total_chunks = 0
    
    for name, data in corpora_data.items():
        print(f"\n{data['name']:25s}:")
        print(f"  Documents:  {data['doc_count']:4d}")
        print(f"  Chunks:     {data['chunk_count']:4d}")
        print(f"  Avg chunks/doc: {data['chunk_count']/data['doc_count']:.1f}")
        
        total_docs += data['doc_count']
        total_chunks += data['chunk_count']
    
    print(f"\n{'='*70}")
    print(f"TOTAL: {total_docs} documents → {total_chunks} chunks")
    print(f"{'='*70}")
    
    # Final cost and time estimates
    estimated_cost = total_chunks * 0.003
    estimated_time_mins = total_chunks * 1.5 / 60
    estimated_time_hrs = estimated_time_mins / 60
    
    print(f"\n💰 FULL EXTRACTION ESTIMATES:")
    print(f"   Total chunks:     {total_chunks:,}")
    print(f"   Estimated cost:   ${estimated_cost:.2f}")
    print(f"   Estimated time:   {estimated_time_mins:.1f} minutes ({estimated_time_hrs:.1f} hours)")
    print(f"   Saves progress:   Every 25 chunks + at completion")
    print(f"   Can resume:       Yes, if interrupted")
    
    return corpora_data, estimated_cost, estimated_time_mins

# Prepare full extraction
full_corpora_data, est_cost, est_time = prepare_full_extraction(df_ni)

In [ ]:
# Cell 18: Pre-execution checklist and final confirmation
def pre_extraction_checklist():
    """
    Final checklist before running full extraction
    """
    print("\n" + "="*70)
    print("🔍 PRE-EXTRACTION CHECKLIST")
    print("="*70)
    
    checklist_items = [
        ("✅ OpenAI API key is set", config.openai_api_key is not None),
        ("✅ Data is loaded and prepared", 'full_corpora_data' in globals()),
        ("✅ Output directories exist", os.path.exists('../graph_data')),
        ("✅ Sufficient disk space", True),  # Assume true
        ("✅ Stable internet connection", True),  # Assume true
    ]
    
    all_good = True
    for item, status in checklist_items:
        status_icon = "✅" if status else "❌"
        print(f"{status_icon} {item}")
        if not status:
            all_good = False
    
    print("\n" + "="*70)
    
    if not all_good:
        print("❌ Please resolve issues before proceeding")
        return False
    
    print("\n⚠️  IMPORTANT NOTES:")
    print("   • This will take approximately 1 hour")
    print("   • Progress is saved every 25 chunks")
    print("   • You can interrupt and resume later")
    print("   • Keep your computer awake during extraction")
    print("   • Check your OpenAI usage limits")
    print("\n💡 TIP: Open a new terminal to monitor API usage at:")
    print("   https://platform.openai.com/usage")
    
    return True

# Run checklist
if pre_extraction_checklist():
    print("\n✅ All systems ready for full extraction!")
else:
    print("\n❌ Please fix issues before proceeding")

In [ ]:
# Cell 19: Execute full extraction with robust error handling
def run_full_extraction(corpora_data):
    """
    Run FULL knowledge graph extraction with progress tracking and error handling
    """
    print("\n" + "="*70)
    print("🚀 STARTING FULL KNOWLEDGE GRAPH EXTRACTION")
    print("="*70)
    
    # Calculate totals
    total_chunks = sum(data['chunk_count'] for data in corpora_data.values())
    estimated_cost = total_chunks * 0.003
    estimated_time_hrs = (total_chunks * 1.5 / 60) / 60
    
    print(f"\n📊 EXTRACTION PLAN:")
    print(f"   Corpora:          {len(corpora_data)}")
    print(f"   Total chunks:     {total_chunks:,}")
    print(f"   Estimated cost:   ${estimated_cost:.2f}")
    print(f"   Estimated time:   ~{estimated_time_hrs:.1f} hours")
    print(f"   Progress saves:   Every 25 chunks")
    
    # Final confirmation
    print("\n" + "="*70)
    response = input("⚠️  START FULL EXTRACTION? This will use API credits. (yes/no): ")
    if response.lower() != 'yes':
        print("❌ Extraction cancelled by user")
        return None
    
    print("\n🎯 Starting full extraction at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("="*70)
    
    start_time = datetime.now()
    all_results = {}
    
    # Process each corpus
    for i, (corpus_name, data) in enumerate(corpora_data.items(), 1):
        print(f"\n{'='*70}")
        print(f"📚 CORPUS {i}/{len(corpora_data)}: {data['name']}")
        print(f"{'='*70}")
        print(f"   Documents: {data['doc_count']}")
        print(f"   Chunks:    {data['chunk_count']}")
        print(f"   Status:    Processing...")
        
        corpus_start_time = datetime.now()
        
        try:
            # Extract knowledge triples (no max_chunks limit for full extraction)
            triples = knowledge_extractor.extract_from_corpus(
                chunks=data['chunks'],
                metadata=data['metadata'],
                corpus_name=data['name'],
                max_chunks=None,  # Process ALL chunks
                save_progress=True  # Save every 25 chunks
            )
            
            all_results[corpus_name] = triples
            
            corpus_elapsed = (datetime.now() - corpus_start_time).total_seconds() / 60
            
            print(f"\n✅ {data['name']} COMPLETE!")
            print(f"   Extracted relations: {len(triples):,}")
            print(f"   Time taken:         {corpus_elapsed:.1f} minutes")
            print(f"   Relations/minute:   {len(triples)/corpus_elapsed:.1f}")
            
            # Show high-confidence relationships
            if triples:
                high_conf = [t for t in triples if t.confidence >= 0.8]
                print(f"   High confidence:    {len(high_conf):,} ({len(high_conf)/len(triples)*100:.1f}%)")
                
                print(f"\n   📋 Top 5 Sample Relations:")
                for j, triple in enumerate(sorted(high_conf, key=lambda x: x.confidence, reverse=True)[:5], 1):
                    print(f"      {j}. {triple.subject} → {triple.predicate} → {triple.object}")
                    print(f"         Confidence: {triple.confidence:.2f}, Types: {triple.subject_type}/{triple.object_type}")
        
        except KeyboardInterrupt:
            print("\n⚠️  EXTRACTION INTERRUPTED BY USER")
            print(f"   Completed corpora: {i-1}/{len(corpora_data)}")
            print(f"   Partial results saved in ../graph_data/")
            raise
        
        except Exception as e:
            print(f"\n❌ ERROR in {data['name']}: {e}")
            print(f"   Continuing with next corpus...")
            all_results[corpus_name] = []
    
    # Final summary
    total_elapsed = (datetime.now() - start_time).total_seconds() / 60
    
    print("\n" + "="*70)
    print("🎉 FULL EXTRACTION COMPLETE!")
    print("="*70)
    
    knowledge_extractor.print_final_summary()
    
    print(f"\n⏱️  TIMING SUMMARY:")
    print(f"   Start time:    {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   End time:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Total time:    {total_elapsed:.1f} minutes ({total_elapsed/60:.2f} hours)")
    print(f"   Actual cost:   ${knowledge_extractor.extraction_stats['total_cost']:.2f}")
    
    # Save final combined results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_path = f"../graph_data/ni_education_FULL_kg_results_{timestamp}.pkl"
    
    with open(combined_path, 'wb') as f:
        pickle.dump(all_results, f)
    
    print(f"\n💾 FINAL RESULTS SAVED:")
    print(f"   Location: {combined_path}")
    print(f"   Size:     {os.path.getsize(combined_path) / 1024 / 1024:.2f} MB")
    
    # Save metadata
    metadata_path = f"../graph_data/ni_education_FULL_metadata_{timestamp}.json"
    metadata = {
        'extraction_date': datetime.now().isoformat(),
        'total_time_minutes': total_elapsed,
        'total_cost': knowledge_extractor.extraction_stats['total_cost'],
        'corpora': {
            name: {
                'doc_count': data['doc_count'],
                'chunk_count': data['chunk_count'],
                'relations_extracted': len(all_results.get(name, []))
            }
            for name, data in corpora_data.items()
        },
        'stats': knowledge_extractor.extraction_stats
    }
    
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"   Metadata: {metadata_path}")
    
    print("\n✅ Full extraction pipeline complete!")
    print("   You can now proceed with analysis and visualization")
    
    return all_results

# Execute full extraction
print("\n🎯 Ready to start FULL extraction")
print("📝 Make sure you've reviewed the estimates above")
print("\n" + "="*70)

# Uncomment the line below when you're ready to run:
full_results = run_full_extraction(full_corpora_data)

In [ ]:
# Cell 21: Resume from interruption (if needed)
def resume_from_interruption():
    """
    If extraction was interrupted, load partial results and continue
    """
    print("🔄 Checking for partial extraction results...")
    
    # Find most recent progress files
    progress_files = glob.glob('../graph_data/kg_triples_*_progress_*.pkl')
    
    if not progress_files:
        print("❌ No progress files found. Start fresh extraction.")
        return None
    
    print(f"✅ Found {len(progress_files)} progress files")
    
    # Load most recent for each corpus
    corpus_results = {}
    
    for corpus_name in ['option1', 'option2', 'both']:
        corpus_files = [f for f in progress_files if corpus_name.upper() in f]
        
        if corpus_files:
            latest = max(corpus_files, key=os.path.getctime)
            print(f"   Loading {corpus_name}: {os.path.basename(latest)}")
            
            with open(latest, 'rb') as f:
                corpus_results[corpus_name] = pickle.load(f)
            
            print(f"      Loaded {len(corpus_results[corpus_name])} relations")
    
    print(f"\n✅ Loaded partial results from {len(corpus_results)} corpora")
    print("💡 You can continue analysis with these partial results")
    print("   or restart extraction from where it stopped")
    
    return corpus_results

# Run if needed:
# partial_results = resume_from_interruption()

In [ ]:
# Cell 22: Neo4j Connection Setup
from neo4j import GraphDatabase
import os

class NIEducationNeo4j:
    """
    Neo4j integration for Northern Ireland education knowledge graph
    """
    
    def __init__(self):
        self.driver = None
        self.corpus_colors = {
            'option1': '#FF6B6B',      # Red for Option1 (Protestant/Unionist)
            'option2': '#4ECDC4',      # Teal for Option2 (Catholic/Nationalist)
            'both': '#95E1D3'          # Green for Both/Shared
        }
        self.connect()
    
    def connect(self):
        """Connect to Neo4j database"""
        try:
            # Get credentials from environment or prompt
            neo4j_uri = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
            neo4j_user = os.getenv('NEO4J_USERNAME', 'neo4j')
            neo4j_password = os.getenv('NEO4J_PASSWORD')
            
            if not neo4j_password:
                print("⚠️  Neo4j password not found in environment")
                neo4j_password = input("Enter Neo4j password: ")
            
            self.driver = GraphDatabase.driver(
                neo4j_uri,
                auth=(neo4j_user, neo4j_password)
            )
            
            # Test connection
            with self.driver.session() as session:
                result = session.run("RETURN 1 as test")
                print("✅ Neo4j connection successful!")
                print(f"   URI: {neo4j_uri}")
                
        except Exception as e:
            print(f"❌ Neo4j connection failed: {e}")
            print("\n💡 Make sure:")
            print("   1. Neo4j is running (neo4j start)")
            print("   2. Correct credentials in .env file")
            print("   3. URI is correct (default: bolt://localhost:7687)")
            self.driver = None
    
    def close(self):
        """Close database connection"""
        if self.driver:
            self.driver.close()
            print("🔌 Neo4j connection closed")

# Initialize Neo4j connection
ni_neo4j = NIEducationNeo4j()

In [ ]:
# Cell 23: Clear and prepare Neo4j database
def clear_neo4j_database(neo4j_handler):
    """
    Clear all existing data from Neo4j database
    """
    if not neo4j_handler.driver:
        print("❌ No Neo4j connection")
        return False
    
    print("🗑️  Clearing Neo4j database...")
    
    response = input("⚠️  This will DELETE ALL data in Neo4j. Continue? (yes/no): ")
    if response.lower() != 'yes':
        print("❌ Operation cancelled")
        return False
    
    with neo4j_handler.driver.session() as session:
        # Delete all nodes and relationships
        session.run("MATCH (n) DETACH DELETE n")
        print("✅ Neo4j database cleared")
    
    return True

def create_neo4j_constraints(neo4j_handler):
    """
    Create constraints and indexes for better performance
    """
    if not neo4j_handler.driver:
        return False
    
    print("📋 Creating constraints and indexes...")
    
    with neo4j_handler.driver.session() as session:
        constraints = [
            "CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (n:Entity) REQUIRE n.name IS UNIQUE",
            "CREATE INDEX entity_type IF NOT EXISTS FOR (n:Entity) ON (n.type)",
            "CREATE INDEX entity_corpus IF NOT EXISTS FOR (n:Entity) ON (n.corpus)",
        ]
        
        for constraint in constraints:
            try:
                session.run(constraint)
                print(f"   ✅ {constraint.split()[1]}")
            except Exception as e:
                print(f"   ⚠️  {constraint.split()[1]} (may already exist)")
    
    print("✅ Constraints and indexes ready")
    return True

# Clear and prepare database
if ni_neo4j.driver:
    clear_neo4j_database(ni_neo4j)
    create_neo4j_constraints(ni_neo4j)

In [ ]:
# Cell 24 (FIXED): Load knowledge graph into Neo4j - Skip null values
def load_kg_to_neo4j_safe(neo4j_handler, results, min_confidence=0.7):
    """
    Load Northern Ireland education knowledge graph into Neo4j
    SAFELY handles null values by filtering them out
    """
    if not neo4j_handler.driver:
        print("❌ No Neo4j connection")
        return False
    
    print("\n📤 Loading knowledge graph into Neo4j (with null filtering)...")
    print("="*70)
    
    total_loaded = 0
    total_filtered = 0
    corpus_stats = {}
    
    with neo4j_handler.driver.session() as session:
        
        for corpus_name, triples in results.items():
            print(f"\n📚 Processing {corpus_name.upper()} corpus...")
            
            # Filter by confidence
            high_conf_triples = [t for t in triples if t.confidence >= min_confidence]
            print(f"   Confidence filter: {len(triples)} → {len(high_conf_triples)} (min: {min_confidence})")
            
            # Filter out null values
            valid_triples = []
            filtered_count = 0
            
            for triple in high_conf_triples:
                # Check for null/empty values
                if (triple.subject and str(triple.subject).strip() and 
                    triple.object and str(triple.object).strip() and
                    triple.predicate and str(triple.predicate).strip() and
                    triple.subject_type and str(triple.subject_type).strip() and
                    triple.object_type and str(triple.object_type).strip()):
                    valid_triples.append(triple)
                else:
                    filtered_count += 1
                    # Debug: print what was filtered
                    if filtered_count <= 3:  # Show first 3 examples
                        print(f"   ⚠️  Filtered null: subject='{triple.subject}', object='{triple.object}'")
            
            print(f"   Null filter: {len(high_conf_triples)} → {len(valid_triples)} (removed {filtered_count})")
            total_filtered += filtered_count
            
            if not valid_triples:
                print(f"   ⚠️  No valid triples after filtering - skipping corpus")
                continue
            
            # Get corpus color and display name
            corpus_color = neo4j_handler.corpus_colors.get(corpus_name, '#888888')
            corpus_display = {
                'option1': 'Option1 (Protestant/Unionist)',
                'option2': 'Option2 (Catholic/Nationalist)',
                'both': 'Both (Shared/Neutral)'
            }.get(corpus_name, corpus_name)
            
            # Collect unique entities (with null checking)
            entities = {}
            for triple in valid_triples:
                # Subject entity
                subject_name = str(triple.subject).strip()
                if subject_name and subject_name not in entities:
                    entities[subject_name] = {
                        'type': str(triple.subject_type).strip(),
                        'occurrences': 0
                    }
                if subject_name:
                    entities[subject_name]['occurrences'] += 1
                
                # Object entity
                object_name = str(triple.object).strip()
                if object_name and object_name not in entities:
                    entities[object_name] = {
                        'type': str(triple.object_type).strip(),
                        'occurrences': 0
                    }
                if object_name:
                    entities[object_name]['occurrences'] += 1
            
            print(f"   Creating {len(entities)} entities...")
            
            # Create entities with error handling
            from tqdm import tqdm
            errors = 0
            for entity_name, entity_data in tqdm(entities.items(), desc="Entities"):
                try:
                    session.run("""
                    MERGE (n:Entity {name: $name})
                    SET n.type = $type,
                        n.corpus = $corpus,
                        n.corpus_display = $corpus_display,
                        n.color = $color,
                        n.occurrences = $occurrences
                    """,
                    name=entity_name,
                    type=entity_data['type'],
                    corpus=corpus_name,
                    corpus_display=corpus_display,
                    color=corpus_color,
                    occurrences=entity_data['occurrences'])
                except Exception as e:
                    errors += 1
                    if errors <= 3:
                        print(f"      ⚠️  Error creating entity '{entity_name}': {e}")
            
            if errors > 0:
                print(f"   ⚠️  {errors} entities failed to create")
            
            print(f"   Creating {len(valid_triples)} relationships...")
            
            # Create relationships with error handling
            rel_errors = 0
            for triple in tqdm(valid_triples, desc="Relations"):
                try:
                    session.run("""
                    MATCH (a:Entity {name: $subject})
                    MATCH (b:Entity {name: $object})
                    MERGE (a)-[r:RELATED]->(b)
                    SET r.type = $relation,
                        r.confidence = $confidence,
                        r.corpus = $corpus,
                        r.corpus_display = $corpus_display,
                        r.color = $color
                    """,
                    subject=str(triple.subject).strip(),
                    object=str(triple.object).strip(),
                    relation=str(triple.predicate).strip(),
                    confidence=triple.confidence,
                    corpus=corpus_name,
                    corpus_display=corpus_display,
                    color=corpus_color)
                except Exception as e:
                    rel_errors += 1
                    if rel_errors <= 3:
                        print(f"      ⚠️  Error creating relationship: {e}")
            
            if rel_errors > 0:
                print(f"   ⚠️  {rel_errors} relationships failed to create")
            
            corpus_stats[corpus_name] = {
                'entities': len(entities),
                'relations': len(valid_triples),
                'filtered': filtered_count
            }
            
            total_loaded += len(valid_triples)
            print(f"   ✅ Successfully loaded {len(valid_triples)} relations")
    
    # Print summary
    print("\n" + "="*70)
    print("📊 NEO4J LOADING SUMMARY")
    print("="*70)
    
    for corpus, stats in corpus_stats.items():
        corpus_display = {
            'option1': 'Option1',
            'option2': 'Option2',
            'both': 'Both'
        }.get(corpus, corpus)
        
        print(f"\n{corpus_display:10s}:")
        print(f"   Entities:  {stats['entities']:,}")
        print(f"   Relations: {stats['relations']:,}")
        print(f"   Filtered:  {stats['filtered']:,}")
    
    print(f"\n{'='*70}")
    print(f"TOTAL LOADED:   {total_loaded:,} relations")
    print(f"TOTAL FILTERED: {total_filtered:,} relations (with null values)")
    print(f"{'='*70}")
    
    # Verify data in Neo4j
    with neo4j_handler.driver.session() as session:
        result = session.run("MATCH (n:Entity) RETURN count(n) as node_count")
        node_count = result.single()['node_count']
        
        result = session.run("MATCH ()-[r:RELATED]->() RETURN count(r) as rel_count")
        rel_count = result.single()['rel_count']
        
        print(f"\n✅ Verification:")
        print(f"   Nodes in Neo4j:         {node_count:,}")
        print(f"   Relationships in Neo4j: {rel_count:,}")
    
    print("\n🎉 Knowledge graph successfully loaded into Neo4j!")
    return True

# Use the safe version to load
if ni_neo4j.driver and 'full_results' in globals():
    load_kg_to_neo4j_safe(ni_neo4j, full_results, min_confidence=0.75)
else:
    print("⚠️  Make sure you have:")
    print("   1. Neo4j connection (run Cell 22)")
    print("   2. Extraction results loaded (full_results variable)")

In [ ]:
# Cell 25: Generate Neo4j Browser styling
def generate_neo4j_styles(neo4j_handler):
    """
    Generate styling commands for Neo4j Browser
    """
    print("\n🎨 Neo4j Browser Styling Guide")
    print("="*70)
    
    print("\n1️⃣  Open Neo4j Browser:")
    print(f"   URL: http://localhost:7474")
    print(f"   (or your Neo4j Browser URL)")
    
    print("\n2️⃣  Click the 'Style' icon (paintbrush) in the bottom left")
    
    print("\n3️⃣  Add this styling configuration:")
    print("\n" + "-"*70)
    
    style_config = """
node {
  diameter: 50px;
  color: #A5ABB6;
  border-color: #9AA1AC;
  border-width: 2px;
  text-color-internal: #FFFFFF;
  font-size: 10px;
}

relationship {
  color: #A5ABB6;
  shaft-width: 1px;
  font-size: 8px;
  padding: 3px;
  text-color-external: #000000;
  text-color-internal: #FFFFFF;
  caption: '<type>';
}

node.Entity {
  color: #data(color);
  border-color: #data(color);
  caption: '{name}';
  diameter: 50px;
}

relationship.RELATED {
  color: #data(color);
  shaft-width: 2px;
  caption: '{type}';
}
"""
    
    print(style_config)
    print("-"*70)
    
    print("\n4️⃣  Apply the styles by clicking 'Apply'")
    
    print("\n✅ Your nodes will now be colored by corpus:")
    print(f"   🔴 Option1 (Protestant/Unionist): {neo4j_handler.corpus_colors['option1']}")
    print(f"   🔵 Option2 (Catholic/Nationalist): {neo4j_handler.corpus_colors['option2']}")
    print(f"   🟢 Both (Shared/Neutral): {neo4j_handler.corpus_colors['both']}")

generate_neo4j_styles(ni_neo4j)